# Step 0 : Queries for RAGAS Evaluation

In [39]:
import os
from dotenv import load_dotenv

# Load FIRST, before any other imports
load_dotenv("/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/ingestion.env")

# Verify it's loaded
print("API Key loaded:", bool(os.getenv("OPENAI_API_KEY")))


API Key loaded: True


In [40]:
questions = [
    "What is the purpose of the SEBI Master Circular for compliance with the LODR Regulations, 2015 by listed entities?",
    "Which earlier master circular is superseded by the November 11, 2024 LODR master circular?",
    "What are the three categories into which holding of specified securities must be divided under the LODR master circular?",
    "How is total shareholding calculated for the purpose of computing public shareholding under the LODR master circular?",
    "How is percentage of promoter shareholding calculated under the LODR master circular?",
    "What details must be disclosed for public shareholders holding 1% or more of shares in a listed entity?",
    "Under what conditions do shares underlying depository receipts form part of public shareholding?",
    "What are the disclosure tables prescribed for the shareholding pattern under Annexure 2 of the LODR master circular?",
    "What exemptions may be considered while determining whether 100% of promoter and promoter group shareholding is in dematerialized form?",
    "What is the minimum dematerialized holding requirement for non-promoter shareholding under the LODR master circular?",
    "What reports must be submitted for compliance with the corporate governance provisions under regulation 27(2) of the LODR Regulations?",
    "What must a listed entity submit if the corporate governance provisions are not applicable to it under the LODR framework?",
    "Within what time must an entity issuing Indian Depository Receipts file the IDR holding pattern with the stock exchange?",
    "What comparative corporate governance analysis must an IDR issuing listed entity submit under regulation 72 of the LODR Regulations?",
    "Within how many days must quarterly and annual financial results be submitted under regulation 33(3) of the LODR Regulations?",
    "What must a listed entity do if it delays submission of financial results beyond the prescribed timeline?",
    "What minimum segment information must be disclosed in quarterly or annual segment reporting under the LODR master circular?",
    "Which entities are covered by the procedure for limited review or audit of accounts consolidated with a listed entity?",
    "In which cases must trading take place in the Trade for Trade segment for the first 10 trading days under the surveillance master circular?",
    "What must stock exchanges ensure before starting trading in scrips covered by merger, demerger, direct listing, or revocation of suspension scenarios?",
    "What internal controls are SEBI registered market intermediaries required to establish regarding unauthenticated news circulation?",
    "What restrictions or supervision requirements apply to employee access to social media, messaging services, blogs, websites, or email under the surveillance master circular?",
    "What is the role of the Compliance Officer when an employee receives market-related news under the surveillance master circular?",
    "What disclosures may be maintained by a company under Regulation 6 of the SEBI (Prohibition of Insider Trading) Regulations, 2015?",
    "What must companies do regarding the Code of Fair Disclosure and Code of Conduct under Regulations 8 and 9 of the PIT Regulations?",
    "How must violations of the Code of Conduct under the PIT Regulations be reported to stock exchanges?",
    "What forms of payment are permitted for remitting amounts to the Investor Protection and Education Fund under the surveillance master circular?",
    "For whom will system driven disclosures under Regulation 7(2) of the PIT Regulations be implemented?",
    "What is the fine for delay in completion of a bonus issue under the ICDR master circular?",
    "What are the key timelines and process requirements for trading of Rights Entitlements in dematerialized form under the ICDR master circular?"
]

In [41]:
import os
import sys

# Get the absolute path to the project root directory
# Assuming the notebook is at rag_pipeline/ragas_eval/dataset.ipynb
# We need to go up two levels to reach the project root.
project_root = os.path.abspath(os.path.join(os.getcwd(), '../../')) # go two levels up

# Add the project root to sys.path
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Now your imports should work
from rag_pipeline.workflow.config import Settings
from rag_pipeline.workflow.service import RAGService
# ... other imports

# Step 1 : Instantiate the workflow

In [42]:
from rag_pipeline.workflow.config import Settings
from rag_pipeline.workflow.service import RAGService
from rag_pipeline.workflow.graph import RAGWorkflow
from rag_pipeline.workflow.node_orchestrator import Nodes
from rag_pipeline.workflow.database.sessions import Database
from rag_pipeline.workflow.repositories.pinecone_repository import PineconeRepository
from rag_pipeline.workflow.embeddings.sentence_transformer_embedding import (
    SentenceTransformerEmbedding,
)
from rag_pipeline.workflow.embeddings.sparse_embedding import (
    SentenceTransformerSparseEmbedding,
)
from rag_pipeline.workflow.llms.ollama_llama import OllamaLLM
from rag_pipeline.workflow.database.db_repositories.conversation_repository import (
    ConversationRepository,
)
from rag_pipeline.workflow.configs.pinecone_config import PineconeConfig
from rag_pipeline.workflow.configs.llm_config import LLMConfig
from rag_pipeline.api.routes import ask_endpoint
from rag_pipeline.workflow.embeddings.openai_embedding import OpenAIEmbedding
from rag_pipeline.workflow.llms.openai import OpenAILLM

from dotenv import load_dotenv
load_dotenv("/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/ingestion.env")
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

settings = Settings()


# Database
database = Database(settings.database_url)
logger.info("Database initialized")

# Embeddings configuration
pinecone_config = PineconeConfig()

# Embedding strategies
"""
dense_embedding = SentenceTransformerEmbedding(pinecone_config)
"""
dense_embedding = OpenAIEmbedding()
sparse_embedding = SentenceTransformerSparseEmbedding(pinecone_config)
logger.info("Embedding strategies initialized")

# Vector Database
vector_db = PineconeRepository(
    api_key=os.getenv("PINECONE_API_KEY"),
    pinecone_config=pinecone_config,
    dense_embedding_strategy=dense_embedding,
    sparse_embedding_strategy=sparse_embedding,
    environment=settings.pinecone_environment,
)

logger.info("Vector database initialized")

# LLM
llm_config = LLMConfig(model_name=settings.openai_model_name)
llm = OpenAILLM(llm_config)

logger.info("LLM initialized")

# Repository
conversation_repo = ConversationRepository()
logger.info("Conversation repository initialized")

# Service
service = RAGService(
    database=database,
    vector_db=vector_db,
    conversation_repository=conversation_repo,
    llm=llm
)
logger.info("RAG service initialized")

# Workflow
nodes = Nodes(service=service)
workflow = RAGWorkflow(nodes=nodes)

2026-03-31 01:42:08,214 - __main__ - INFO - Database initialized
2026-03-31 01:42:08,224 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: mps
2026-03-31 01:42:08,225 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SparseEncoder: naver/splade-cocondenser-ensembledistil
2026-03-31 01:42:08,549 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/naver/splade-cocondenser-ensembledistil/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-03-31 01:42:08,563 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/naver/splade-cocondenser-ensembledistil/49cf4c7b0db5b870a401ddf5e2669993ef3699c7/modules.json "HTTP/1.1 200 OK"
2026-03-31 01:42:08,848 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/naver/splade-cocondenser-ensembledistil/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-31 01:42:08,873 - httpx - INFO - HTTP Request: HEAD https://huggin

# Step 2 : Create the raw form of raga dataset

In [43]:
"""
dataset = [
    {
        "user_input": "What is the refund policy?",
        "retrieved_contexts": [
            "Refunds are allowed within 30 days of purchase.",
            "Refunds are processed to the original payment method."
        ],
        "response": "You can get a refund within 30 days, and it will be sent to your original payment method."
    },
    {
        "user_input": "Can I upgrade my plan?",
        "retrieved_contexts": [
            "Users can upgrade their subscription at any time.",
        ],
        "response": "Yes, you can upgrade anytime."
    }
]
"""

'\ndataset = [\n    {\n        "user_input": "What is the refund policy?",\n        "retrieved_contexts": [\n            "Refunds are allowed within 30 days of purchase.",\n            "Refunds are processed to the original payment method."\n        ],\n        "response": "You can get a refund within 30 days, and it will be sent to your original payment method."\n    },\n    {\n        "user_input": "Can I upgrade my plan?",\n        "retrieved_contexts": [\n            "Users can upgrade their subscription at any time.",\n        ],\n        "response": "Yes, you can upgrade anytime."\n    }\n]\n'

In [44]:
def raga_dataset_creation(workflow, questions):
    ragas_raw_dataset = []
    for question in questions:
        response = workflow.execute(question, session_id = 1)

        dict1= {
            "user_inputs" : response["query"],
            "retrieved_contexts" : [doc.page_content for doc in response["retrieved_documents"]],
            "response" : response["response"]
        }
        ragas_raw_dataset.append(dict1)
    
    return ragas_raw_dataset

ragas_raw_dataset = raga_dataset_creation(workflow, questions)


2026-03-31 01:42:14,887 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-31 01:42:15,110 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00, 15.24it/s]
2026-03-31 01:42:17,070 - rag_pipeline.workflow.service - INFO - Retrieved 10 documents for query
2026-03-31 01:42:18,024 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-31 01:42:22,969 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-31 01:42:22,970 - rag_pipeline.workflow.service - INFO - Extracted response content (length: 668): The purpose of the SEBI Master Circular for compliance with the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015 (LODR Regulations) for listed...
2026-03-31 01:42:22,980 - rag_pipeline.workflow.service - 

In [45]:
ragas_raw_dataset[0]

{'user_inputs': 'What is the purpose of the SEBI Master Circular for compliance with the LODR Regulations, 2015 by listed entities?',
 'retrieved_contexts': ['Page 1 of 261 \n \nMASTER CIRCULAR \n                                                                                                           SEBI/HO/CFD/PoD2/CIR/P/0155\n                                                                                    November 11, 2024 \n \nTo \n \nAll listed entities1 \nAll Recognized Stock Exchanges \nAll the Depositories \nOther Stakeholders2 \n \nMadam / Sir, \n \nSub:  Master circular for compliance with the provisions of the Securities and Exchange \nBoard of India (Listing Obligations and Disclosure Requirements) Regul ations, \n2015 by listed entities  \n \n1. The Securities and Exchange Board of India had notified the Securities and Exchange Board of \nIndia (Listing Obligations and Disclosure Requirements) Regulations, 2015 (hereinafter referred to \nas “LODR Regulations”) which ca

# Step 3 : Convert to RAGAS Objects 

In [46]:
from ragas import EvaluationDataset, SingleTurnSample

samples = []

for row in ragas_raw_dataset:
    samples.append(
        SingleTurnSample(
            user_input=row["user_inputs"],
            response=row["response"],
            retrieved_contexts=row["retrieved_contexts"]
        )
    )

eval_dataset = EvaluationDataset(samples=samples)

# Step 4 : Start Ragas Evaluation with local llm and embeddings

In [47]:
import os

from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Ensure API key exists
assert os.getenv("OPENAI_API_KEY")

# ---- Create default LLM & embeddings ----
llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4o-mini")   # default-ish lightweight model
)

embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings()
)


/var/folders/lz/6vnq1xf93h912vrdhnbwvvmc0000gn/T/ipykernel_93458/662960238.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy
/var/folders/lz/6vnq1xf93h912vrdhnbwvvmc0000gn/T/ipykernel_93458/662960238.py:4: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy
/var/folders/lz/6vnq1xf93h912vrdhnbwvvmc0000gn/T/ipykernel_93458/662960238.py:15: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gp

In [48]:
result = evaluate(
    dataset=eval_dataset,
    metrics=[
        Faithfulness(llm=llm),
        AnswerRelevancy(llm=llm, embeddings=embeddings),
    ],
    show_progress=True,
)

print(result)


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]2026-03-31 01:47:40,949 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-31 01:47:40,956 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-31 01:47:42,300 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-31 01:47:43,145 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Evaluating:   2%|▏         | 1/60 [00:04<03:57,  4.02s/it]2026-03-31 01:47:43,688 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-31 01:47:43,690 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-31 01:47:43,692 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-31 01:47:43,694 - httpx - INFO - HTTP Request: PO

{'faithfulness': 0.8987, 'answer_relevancy': 0.9579}


In [49]:
print(result)

{'faithfulness': 0.8987, 'answer_relevancy': 0.9579}
